In [11]:
! git clone "https://github.com/amirrezafateh/Multi-Scale-Transformer-Pneumonia.git"

Cloning into 'Multi-Scale-Transformer-Pneumonia'...
remote: Enumerating objects: 286, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 286 (delta 68), reused 0 (delta 0), pack-reused 162 (from 1)
Receiving objects: 100% (286/286), 2.79 MiB | 15.05 MiB/s, done.
Resolving deltas: 100% (144/144), done.


In [12]:
%%writefile /kaggle/working/Multi-Scale-Transformer-Pneumonia/data/data_loader_segmentation.py

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx].replace('.jpg', '.png'))

        image = Image.open(img_path).convert("RGB")  # Ensure RGB scale
        mask = Image.open(mask_path).convert("L")  # Ensure binary

        if self.transform is not None:
            image, mask = self.transform(image, mask)

        return image, mask

class JointTransform:
    def __init__(self, base_transform, mask_transform=None):
        self.base_transform = base_transform
        self.mask_transform = mask_transform if mask_transform else base_transform

    def __call__(self, image, mask):
        seed = random.randint(0, 2**32)
        random.seed(seed)
        torch.manual_seed(seed)
        image = self.base_transform(image)

        random.seed(seed)
        torch.manual_seed(seed)
        mask = self.mask_transform(mask)

        return image, mask

def get_data_loader(data_dir, batch_size):
    base_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.16666122, 0.16666122, 0.16666122], std = [0.17487395, 0.17487395, 0.17487395])
    ])

    mask_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
    ])

    train_transform = JointTransform(base_transform, mask_transform)

    test_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.16666122, 0.16666122, 0.16666122], std = [0.17487395, 0.17487395, 0.17487395])
    ])

    test_mask_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])
    
    train_dataset = SegmentationDataset(
        image_dir=os.path.join(data_dir, "train", "images"),
        mask_dir=os.path.join(data_dir, "train", "masks"),
        transform=train_transform
    )

    test_dataset = SegmentationDataset(
        image_dir=os.path.join(data_dir, "test", "images"),
        mask_dir=os.path.join(data_dir, "test", "masks"),
        transform=JointTransform(test_transform, test_mask_transform)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    return train_loader, test_loader

Overwriting /kaggle/working/Multi-Scale-Transformer-Pneumonia/data/data_loader_segmentation.py


In [13]:
%%writefile /kaggle/working/Multi-Scale-Transformer-Pneumonia/train_seg.py

import argparse
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from model.UNet import UNet  # Ensure UNet class is defined in model/unet.py
from data.data_loader_segmentation import get_data_loader
from torchvision.utils import save_image

def dice_coefficient(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    target = target.float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    
    if union.item() == 0:
        return 1.0
    
    return 2.0 * intersection / union


def count_parameters(model, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    else:
        return sum(p.numel() for p in model.parameters())

def train(model, train_loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0  # Track Dice coefficient

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Compute Dice coefficient
        dice = dice_coefficient(outputs, targets)
        running_dice += dice.item()

        if batch_idx % 100 == 0:
            print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {running_loss/(batch_idx+1):.3f}, Dice: {running_dice/(batch_idx+1):.3f}')

    train_loss = running_loss / len(train_loader)
    train_dice = running_dice / len(train_loader)  # Average Dice coefficient
    return train_loss, train_dice


def validate(model, val_loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0  # Track Dice coefficient
    val_outputs = []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            val_outputs.append(outputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()

            # Compute Dice coefficient
            dice = dice_coefficient(outputs, targets)
            running_dice += dice.item()
    # Save validation output masks
    save_dir = 'validation_output_masks'
    os.makedirs(save_dir, exist_ok=True)
    for i, output in enumerate(val_outputs):
        save_image(output, os.path.join(save_dir, f'epoch_{epoch}_batch_{i}_output.png'))

    val_loss = running_loss / len(val_loader)
    val_dice = running_dice / len(val_loader)  # Average Dice coefficient
    return val_loss, val_dice


def main(args):
    print(":=========== Pneumonia Chest X-ray Segmentation ===========")
    print(f"|             datapath: {args.data_dir}")
    print(f"|              logpath: {args.logpath}")
    print(f"|                  bsz: {args.bsz}")
    print(f"|                   lr: {args.lr}")
    print(f"|                niter: {args.niter}")
    print(":==========================================================")

    # Get the data loaders for the Pneumonia dataset
    train_loader, val_loader = get_data_loader(args.data_dir, args.bsz)

    # Instantiate the UNet model
    model = UNet(in_channels=3, out_channels=1, feature_dims=64)  # UNet for binary segmentation

    # Print the number of parameters
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    print(f"\nTotal # param.: {total_params}")
    print(f"Learnable # param.: {trainable_params}\n")

    # Move the model to the GPU if available
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model = torch.nn.DataParallel(model, device_ids = [0, 1])
    
    # Define loss function and optimizer
    criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)

    scheduler = CosineAnnealingLR(optimizer, T_max=args.niter, eta_min=1e-6)
    
    best_val_loss = float('inf')

    # Train the model
    for epoch in range(args.niter):  # Train for the specified number of epochs
        print(f'Epoch {epoch+1}/{args.niter}')
        start_time = time.time()
        
        train_loss, train_dice = train(model, train_loader, optimizer, criterion, device)
        val_loss, val_dice = validate(model, val_loader, criterion, device, epoch)
        
        scheduler.step()
        
        end_time = time.time()
        epoch_time = end_time - start_time
        
        print(f'Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}')
        print(f'Validation Loss: {val_loss:.4f}, Validation Dice: {val_dice:.4f}')
        print(f'Epoch {epoch+1} completed in {epoch_time:.2f} seconds\n')
        
        # Save the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            os.makedirs(args.logpath, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(args.logpath, 'best_model.pth'))
            print(f'Best model saved with loss: {best_val_loss:.4f}')


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train UNet model on Pneumonia chest X-ray segmentation dataset")
    parser.add_argument('--bsz', type=int, default=8, help='Batch size for training')
    parser.add_argument('--lr', type=float, default=1e-3, help='Learning rate for optimizer')
    parser.add_argument('--niter', type=int, default=30, help='Number of training iterations (epochs)')
    parser.add_argument('--data_dir', type=str, default='segmentation', help='Directory where dataset is located')
    parser.add_argument('--logpath', type=str, default='log', help='Directory to save the best model checkpoint')
    
    args = parser.parse_args()
    main(args)

Overwriting /kaggle/working/Multi-Scale-Transformer-Pneumonia/train_seg.py


In [14]:
! python "/kaggle/working/Multi-Scale-Transformer-Pneumonia/train_seg.py" --data_dir "/kaggle/input/brisc2025/segmentation_task" --bsz 32 --niter 50 --lr 0.0001

:=========== Pneumonia Chest X-ray Segmentation ===========
|             datapath: /kaggle/input/brisc2025/segmentation_task
|              logpath: log
|                  bsz: 32
|                   lr: 0.0001
|                niter: 50
:==========================================================

Total # param.: 43967489
Learnable # param.: 43967489

Epoch 1/50
Batch 0/123, Loss: 0.922, Dice: 0.046
Batch 100/123, Loss: 0.822, Dice: 0.134
Train Loss: 0.8162, Train Dice: 0.1376
Validation Loss: 0.7907, Validation Dice: 0.1049
Epoch 1 completed in 184.18 seconds

Best model saved with loss: 0.7907
Epoch 2/50
Batch 0/123, Loss: 0.789, Dice: 0.113
Batch 100/123, Loss: 0.778, Dice: 0.190
Train Loss: 0.7767, Train Dice: 0.2015
Validation Loss: 0.7694, Validation Dice: 0.3539
Epoch 2 completed in 188.82 seconds

Best model saved with loss: 0.7694
Epoch 3/50
Batch 0/123, Loss: 0.768, Dice: 0.259
Batch 100/123, Loss: 0.760, Dice: 0.355
Train Loss: 0.7589, Train Dice: 0.3715
Validation Loss: 0.